# Funnel system — provenance-routed fake-news detection

**The system under test:** a RoBERTa AI-vs-Human **router** (§7m) sends each article either to
**paper-faithful LIFE** (BCE head, §7e — the specialist for LLM-generated news) or to a **plain BERT**
(§7o/§7q — the specialist for human news). The whole thing is scored as ONE binary fake-vs-real
classifier (fake = HF+MF = 1, real = HR+MR = 0) on a shared held-out test set.

**Why:** LIFE detects LLM-*generation*, not fakeness (§7j negative control), while a content BERT
does separate human fake from real (§7q). Routing each article to the expert that actually has
signal is the constructive synthesis — IF it beats doing nothing clever.

| variant | routing | machine branch | question it answers |
|---|---|---|---|
| funnel | RoBERTa router | LIFE | the system as proposed |
| oracle | true provenance | LIFE | what do the router's errors cost? |
| two_bert | RoBERTa router | plain BERT | does LIFE earn its branch in-system? |
| oracle_two_bert | true provenance | plain BERT | ceiling of the two-BERT variant |
| monolithic | none | — | does routing add anything at all? |

**The split** (`funnel_system/make_split.py`): one 70/15/15 train/val/test split per seed (7/42/123),
shared by every component. MF/MR are GPT-3.5 rewrites of HF/HR stories and share their ids, so the
split is made over **id-groups** — a story and all its rewrites land on the same side, otherwise every
veracity model would train on the content of test articles.

Run order: top to bottom. PolitiFact++ first (pilot, ~85-article test — plumbing validation only),
then GossipCop++ (~3,078-article test — the citable result).

In [3]:
# GPU + deps (transformers, fastNLP etc. for the LIFE branch)
!nvidia-smi
%pip install -q -r requirements.txt

Tue Jul 28 18:54:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/LIFE'

DATASET_ROOT = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset'
POLITIFACT_DIR = f'{DATASET_ROOT}/PolitiFact++'
GOSSIPCOP_DIR = f'{DATASET_ROOT}/GossipCop++'

# LIFE features already on Drive (LLaMA2-7B reconstruction; see the inventory cell)
PF_FEATURES = f'{PROJECT_DIR}/dataset/features_llama_multi'            # all four subsets, 520 articles
GC_FEATURES_MULTI = f'{PROJECT_DIR}/dataset/gossipcop/features_llama_multi'  # all four (if the 4-class track ran)
GC_FEATURES_BIN = f'{PROJECT_DIR}/dataset/gossipcop/features_llama'          # MF/MR only (binary track)

# Everything this notebook produces lands under funnel_system/
PF_RUN = f'{PROJECT_DIR}/funnel_system/run_politifact'
GC_RUN = f'{PROJECT_DIR}/funnel_system/run_gossipcop'
GC_MISROUTE_RAW = f'{PROJECT_DIR}/funnel_system/gc_misroute_raw'
GC_MISROUTE_FEATURES = f'{PROJECT_DIR}/funnel_system/gc_misroute_features'
for d in (PF_RUN, GC_RUN):
    os.makedirs(d, exist_ok=True)

os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

Mounted at /content/drive
cwd: /content/drive/MyDrive/LIFE


In [6]:
# Feature inventory — decides the GossipCop LIFE coverage path.
# Expected if complete: PF_FEATURES ~ HF 97 / HR 194 / MF 97 / MR 132 (minus a few drops);
# GC_FEATURES_BIN ~ MF 4084 / MR 4169; GC_FEATURES_MULTI additionally HF 4084 / HR 8168.
import os

def inventory(d):
    if not os.path.isdir(d):
        print(f'MISSING   {d}')
        return
    print(f'exists    {d}')
    for fn in sorted(os.listdir(d)):
        if fn.endswith('.jsonl'):
            with open(os.path.join(d, fn), encoding='utf-8') as f:
                n = sum(1 for _ in f)
            print(f'          {fn}: {n} records')

for d in (PF_FEATURES, GC_FEATURES_MULTI, GC_FEATURES_BIN):
    inventory(d)

exists    /content/drive/MyDrive/LIFE/dataset/features_llama_multi
          HF_fake.jsonl: 97 records
          HR_true.jsonl: 194 records
          MF_fake.jsonl: 97 records
          MR_true.jsonl: 132 records
exists    /content/drive/MyDrive/LIFE/dataset/gossipcop/features_llama_multi
          HF_fake.jsonl: 4084 records
          HR_true.jsonl: 8168 records
          MF_fake.jsonl: 4084 records
          MR_true.jsonl: 4169 records
exists    /content/drive/MyDrive/LIFE/dataset/gossipcop/features_llama
          MF_fake.jsonl: 4084 records
          MR_true.jsonl: 4169 records


## PolitiFact++ — pilot (plumbing validation)

~85-article test sets: every number here is ±5pp noise. The pilot exists to prove the split/join/
prediction plumbing end to end before spending GossipCop compute — do NOT read conclusions from it.
LIFE features for all 520 articles already exist (`features_llama_multi`, reused by §7j), so this
section needs no feature generation.

In [7]:
# Unified id-group split, seeds 7/42/123
!python funnel_system/make_split.py --data_dir "{POLITIFACT_DIR}" --out_dir "{PF_RUN}" --name PolitiFact++

[PolitiFact++] 520 articles in 290 id-groups

seed 7 -> /content/drive/MyDrive/LIFE/funnel_system/run_politifact/split_seed7.json
[PolitiFact++] articles per subset x split:
  HF: train=    67 val=    14 test=    16
  HR: train=   134 val=    29 test=    31
  MF: train=    67 val=    14 test=    16
  MR: train=    90 val=    19 test=    23
  test: 86 articles | fake 32 (37.2%) | AI 39 (45.3%)

seed 42 -> /content/drive/MyDrive/LIFE/funnel_system/run_politifact/split_seed42.json
[PolitiFact++] articles per subset x split:
  HF: train=    67 val=    14 test=    16
  HR: train=   134 val=    29 test=    31
  MF: train=    67 val=    14 test=    16
  MR: train=    92 val=    19 test=    21
  test: 84 articles | fake 32 (38.1%) | AI 37 (44.0%)

seed 123 -> /content/drive/MyDrive/LIFE/funnel_system/run_politifact/split_seed123.json
[PolitiFact++] articles per subset x split:
  HF: train=    67 val=    14 test=    16
  HR: train=   134 val=    29 test=    31
  MF: train=    67 val=    14 test

In [8]:
# Router (RoBERTa AI-vs-Human, §7m config)
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --role router --name PolitiFact++ --out_dir "{PF_RUN}"

[PolitiFact++ | router | seed=7] train=349 val=76 test=86 (full shared test) | model=roberta-base label=provenance device=cuda
config.json: 100% 481/481 [00:00<00:00, 2.27MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 126kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 4.71MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 8.64MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 3.76MB/s]

model.safetensors: downloading bytes:  28% 140M/499M [00:01<00:02, 134MB/s, 6.38MB/s  ]  
model.safetensors: downloading bytes:  63% 313M/499M [00:02<00:00, 267MB/s, 24.4MB/s  ]
model.safetensors: reconstructing file:  81% 402M/499M [00:02<00:00, 217MB/s, 6.45MB/s  ]
model.safetensors: downloading bytes: 100% 335M/335M [00:02<00:00, 127MB/s, 31.2MB/s  ]
model.safetensors: reconstructing file: 100% 499M/499M [00:02<00:00, 189MB/s, 46.7MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 5013.85it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                 

In [12]:
# Human expert (plain BERT on HF/HR, §7o/§7q config, best-val)
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --role human_expert --name PolitiFact++ --out_dir "{PF_RUN}"

[PolitiFact++ | human_expert | seed=7] train=192 val=43 test=86 (full shared test) | model=bert-base-uncased label=veracity device=cuda
config.json: 100% 570/570 [00:00<00:00, 2.29MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 224kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 2.58MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 4.61MB/s]

model.safetensors: downloading bytes:  49% 215M/440M [00:01<00:00, 230MB/s, 12.5MB/s  ]
model.safetensors: downloading bytes:  84% 372M/440M [00:02<00:00, 349MB/s, 30.7MB/s  ]
model.safetensors: reconstructing file:  46% 201M/440M [00:02<00:02, 108MB/s, 6.44MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 162MB/s, 38.3MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 172MB/s, 41.4MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5449.92it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
------

In [13]:
# Machine-branch BERT control (plain BERT on MF/MR)
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --role machine_bert --name PolitiFact++ --out_dir "{PF_RUN}"

[PolitiFact++ | machine_bert | seed=7] train=157 val=33 test=86 (full shared test) | model=bert-base-uncased label=veracity device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 4029.71it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archite

In [14]:
# Monolithic control (one BERT on all four subsets, no routing)
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}"
!python funnel_system/run_component.py --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --role monolithic --name PolitiFact++ --out_dir "{PF_RUN}"

[PolitiFact++ | monolithic | seed=7] train=349 val=76 test=86 (full shared test) | model=bert-base-uncased label=veracity device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 3551.86it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architect

In [9]:
# Assemble LIFE train/test feature files for each seed's split
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed7.json" --out_dir "{PF_RUN}"
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed42.json" --out_dir "{PF_RUN}"
!python funnel_system/assemble_life_data.py --features_dir "{PF_FEATURES}" --data_dir "{POLITIFACT_DIR}" --split_json "{PF_RUN}/split_seed123.json" --out_dir "{PF_RUN}"

[seed 7] LIFE data assembled: train=157 (MF/MR) test=86
  HF: features 97/97 | test missing 0/16
  HR: features 194/194 | test missing 0/31
  MF: features 97/97 | test missing 0/16
  MR: features 132/132 | test missing 0/23
[seed 42] LIFE data assembled: train=159 (MF/MR) test=84
  HF: features 97/97 | test missing 0/16
  HR: features 194/194 | test missing 0/31
  MF: features 97/97 | test missing 0/16
  MR: features 132/132 | test missing 0/21
[seed 123] LIFE data assembled: train=158 (MF/MR) test=85
  HF: features 97/97 | test missing 0/16
  HR: features 194/194 | test missing 0/31
  MF: features 97/97 | test missing 0/16
  MR: features 132/132 | test missing 0/22


In [11]:
# LIFE branch (paper-faithful BCE head) per seed. Per-epoch printed metrics are over the
# FULL mixed test set - ignore them; the LIFE anchor comes from evaluate_system.py.
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed7.jsonl" --test_path "{PF_RUN}/life_test_seed7.jsonl" --num_train_epochs 50 --seed 7 --pred_out "{PF_RUN}/life_preds_seed7.jsonl"
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed42.jsonl" --test_path "{PF_RUN}/life_test_seed42.jsonl" --num_train_epochs 50 --seed 42 --pred_out "{PF_RUN}/life_preds_seed42.jsonl"
!python LIFE_train/train_bce.py --train_path "{PF_RUN}/life_train_seed123.jsonl" --test_path "{PF_RUN}/life_test_seed123.jsonl" --num_train_epochs 50 --seed 123 --pred_out "{PF_RUN}/life_preds_seed123.jsonl"

100% 157/157 [00:00<00:00, 7337.85it/s]
100% 86/86 [00:00<00:00, 8328.37it/s]
seed: 7
--------------------------------BCE head (paper Eq 11-12)--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/5 [00:00<?, ?it/s]
Iteration:  20% 1/5 [00:00<00:02,  1.77it/s]
Iteration:  60% 3/5 [00:00<00:00,  4.55it/s]
Iteration: 100% 5/5 [00:00<00:00,  5.20it/s]
epoch 1: train_loss 0.7342119932174682

Iteration:   0% 0/3 [00:00<?, ?it/s]
Iteration:  33% 1/3 [00:00<00:00,  8.34it/s]
Iteration: 100% 3/3 [00:00<00:00, 13.01it/s]
******** Evalation ********
Accuracy: 37.2
Macro F1 Score: 27.1
Precision/Recall per class (real=0, fake=1): 
0.0/0.0 37.2/100.0
************************************************************************************************************************
Epoch:   2% 1/50 [00:01<00:59,  1.22s/it]
Iteration:   0% 0/5 [00:00<?, ?it/s]
Iteration:  20% 1/5 [00:00<00:00,  9.97it/s]
Iteration:  60% 3/5 [00:00<00:00, 10.19it/s]
Iteration:

In [15]:
# Score every system variant (pilot numbers - plumbing validation only)
!python funnel_system/evaluate_system.py --run_dir "{PF_RUN}" --name PolitiFact++


--- seed 7 (86 test articles, always-majority baseline 0.6279) ---
  funnel           acc=0.7558 macroF1=0.7193 fake P/R 0.739/0.531  (65/86)
  oracle           acc=0.7907 macroF1=0.7659 fake P/R 0.769/0.625  (68/86)
  two_bert         acc=0.8023 macroF1=0.7898 fake P/R 0.727/0.750  (69/86)
  oracle_two_bert  acc=0.8023 macroF1=0.7922 fake P/R 0.714/0.781  (69/86)
  monolithic       acc=0.8256 macroF1=0.8121 fake P/R 0.774/0.750  (71/86)

--- seed 42 (84 test articles, always-majority baseline 0.6190) ---
  funnel           acc=0.8571 macroF1=0.8393 fake P/R 0.917/0.688  (72/84)
  oracle           acc=0.8571 macroF1=0.8393 fake P/R 0.917/0.688  (72/84)
  two_bert         acc=0.8810 macroF1=0.8738 fake P/R 0.844/0.844  (74/84)
  oracle_two_bert  acc=0.8810 macroF1=0.8738 fake P/R 0.844/0.844  (74/84)
  monolithic       acc=0.8214 macroF1=0.8041 fake P/R 0.815/0.688  (69/84)

--- seed 123 (85 test articles, always-majority baseline 0.6235) ---
  funnel           acc=0.9176 macroF1=0.907

## GossipCop++ — the citable result

~3,078-article shared test set per seed. The assemble cell picks `features_llama_multi` (all four
subsets) if it exists, else the binary `features_llama` (MF/MR only). With binary-only features,
human test articles have no LIFE features — fine for `oracle`/`two_bert`, and `funnel` falls back to
the human expert for any misrouted human article (counts reported). To close that gap properly, see
the **featurize misrouted articles** section at the bottom, then re-run assemble → LIFE → evaluate.

In [16]:
!python funnel_system/make_split.py --data_dir "{GOSSIPCOP_DIR}" --out_dir "{GC_RUN}" --name GossipCop++

[GossipCop++] 20505 articles in 12252 id-groups

seed 7 -> /content/drive/MyDrive/LIFE/funnel_system/run_gossipcop/split_seed7.json
[GossipCop++] articles per subset x split:
  HF: train=  2858 val=   613 test=   613
  HR: train=  5717 val=  1225 test=  1226
  MF: train=  2858 val=   613 test=   613
  MR: train=  2918 val=   625 test=   626
  test: 3078 articles | fake 1226 (39.8%) | AI 1239 (40.3%)

seed 42 -> /content/drive/MyDrive/LIFE/funnel_system/run_gossipcop/split_seed42.json
[GossipCop++] articles per subset x split:
  HF: train=  2858 val=   613 test=   613
  HR: train=  5717 val=  1225 test=  1226
  MF: train=  2858 val=   613 test=   613
  MR: train=  2918 val=   625 test=   626
  test: 3078 articles | fake 1226 (39.8%) | AI 1239 (40.3%)

seed 123 -> /content/drive/MyDrive/LIFE/funnel_system/run_gossipcop/split_seed123.json
[GossipCop++] articles per subset x split:
  HF: train=  2858 val=   613 test=   613
  HR: train=  5717 val=  1225 test=  1226
  MF: train=  2858 val=  

In [17]:
# Router (RoBERTa AI-vs-Human, §7m config)
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --role router --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --role router --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --role router --name GossipCop++ --out_dir "{GC_RUN}"

[GossipCop++ | router | seed=7] train=13661 val=3076 test=3078 (full shared test) | model=roberta-base label=provenance device=cuda
Loading weights: 100% 197/197 [00:00<00:00, 6111.79it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
  epoch 1: 

In [18]:
# Human expert (plain BERT on HF/HR, §7o/§7q config, best-val)
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --role human_expert --name GossipCop++ --out_dir "{GC_RUN}"

[GossipCop++ | human_expert | seed=7] train=7898 val=1838 test=3078 (full shared test) | model=bert-base-uncased label=veracity device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 6447.89it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arc

In [19]:
# Machine-branch BERT control (plain BERT on MF/MR)
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --role machine_bert --name GossipCop++ --out_dir "{GC_RUN}"

[GossipCop++ | machine_bert | seed=7] train=5775 val=1238 test=3078 (full shared test) | model=bert-base-uncased label=veracity device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 6053.70it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arc

In [20]:
# Monolithic control (one BERT on all four subsets, no routing)
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}"
!python funnel_system/run_component.py --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --role monolithic --name GossipCop++ --out_dir "{GC_RUN}"

[GossipCop++ | monolithic | seed=7] train=13661 val=3076 test=3078 (full shared test) | model=bert-base-uncased label=veracity device=cuda
Loading weights: 100% 199/199 [00:00<00:00, 4715.20it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

In [21]:
# Assemble LIFE data. Prefers the 4-class features (full coverage); falls back to the
# binary MF/MR features. Re-running this cell also picks up on-demand misroute features.
import os
GC_FEATURES = GC_FEATURES_MULTI if os.path.isdir(GC_FEATURES_MULTI) else GC_FEATURES_BIN
GC_FEAT_ARGS = f'--features_dir "{GC_FEATURES}"'
if os.path.isdir(GC_MISROUTE_FEATURES) and os.listdir(GC_MISROUTE_FEATURES):
    GC_FEAT_ARGS += f' --features_dir "{GC_MISROUTE_FEATURES}"'
print('using:', GC_FEAT_ARGS)
!python funnel_system/assemble_life_data.py {GC_FEAT_ARGS} --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed7.json" --out_dir "{GC_RUN}"
!python funnel_system/assemble_life_data.py {GC_FEAT_ARGS} --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed42.json" --out_dir "{GC_RUN}"
!python funnel_system/assemble_life_data.py {GC_FEAT_ARGS} --data_dir "{GOSSIPCOP_DIR}" --split_json "{GC_RUN}/split_seed123.json" --out_dir "{GC_RUN}"

using: --features_dir "/content/drive/MyDrive/LIFE/dataset/gossipcop/features_llama_multi"
[seed 7] LIFE data assembled: train=5776 (MF/MR) test=3078
  HF: features 4084/4084 | test missing 0/613
  HR: features 8168/8168 | test missing 0/1226
  MF: features 4084/4084 | test missing 0/613
  MR: features 4169/4169 | test missing 0/626
[seed 42] LIFE data assembled: train=5776 (MF/MR) test=3078
  HF: features 4084/4084 | test missing 0/613
  HR: features 8168/8168 | test missing 0/1226
  MF: features 4084/4084 | test missing 0/613
  MR: features 4169/4169 | test missing 0/626
[seed 123] LIFE data assembled: train=5776 (MF/MR) test=3078
  HF: features 4084/4084 | test missing 0/613
  HR: features 8168/8168 | test missing 0/1226
  MF: features 4084/4084 | test missing 0/613
  MR: features 4169/4169 | test missing 0/626


In [22]:
# LIFE branch per seed (see the PolitiFact note about per-epoch printed metrics)
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed7.jsonl" --test_path "{GC_RUN}/life_test_seed7.jsonl" --num_train_epochs 8 --seed 7 --pred_out "{GC_RUN}/life_preds_seed7.jsonl"
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed42.jsonl" --test_path "{GC_RUN}/life_test_seed42.jsonl" --num_train_epochs 8 --seed 42 --pred_out "{GC_RUN}/life_preds_seed42.jsonl"
!python LIFE_train/train_bce.py --train_path "{GC_RUN}/life_train_seed123.jsonl" --test_path "{GC_RUN}/life_test_seed123.jsonl" --num_train_epochs 8 --seed 123 --pred_out "{GC_RUN}/life_preds_seed123.jsonl"

Streaming output truncated to the last 5000 lines.
Iteration:  18% 32/181 [00:03<00:15,  9.38it/s]
Iteration:  18% 33/181 [00:03<00:15,  9.35it/s]
Iteration:  19% 34/181 [00:03<00:15,  9.39it/s]
Iteration:  19% 35/181 [00:03<00:15,  9.29it/s]
Iteration:  20% 36/181 [00:03<00:15,  9.24it/s]
Iteration:  20% 37/181 [00:03<00:15,  9.44it/s]
Iteration:  21% 38/181 [00:04<00:15,  9.49it/s]
Iteration:  22% 39/181 [00:04<00:14,  9.49it/s]
Iteration:  22% 40/181 [00:04<00:14,  9.51it/s]
Iteration:  23% 41/181 [00:04<00:14,  9.54it/s]
Iteration:  23% 42/181 [00:04<00:14,  9.47it/s]
Iteration:  24% 43/181 [00:04<00:14,  9.47it/s]
Iteration:  24% 44/181 [00:04<00:14,  9.51it/s]
Iteration:  25% 45/181 [00:04<00:14,  9.49it/s]
Iteration:  25% 46/181 [00:04<00:14,  9.46it/s]
Iteration:  27% 48/181 [00:05<00:13,  9.67it/s]
Iteration:  27% 49/181 [00:05<00:13,  9.55it/s]
Iteration:  28% 50/181 [00:05<00:13,  9.58it/s]
Iteration:  28% 51/181 [00:05<00:13,  9.55it/s]
Iteration:  29% 52/181 [00:05<00:13, 

In [23]:
# The result table
!python funnel_system/evaluate_system.py --run_dir "{GC_RUN}" --name GossipCop++


--- seed 7 (3078 test articles, always-majority baseline 0.6017) ---
  funnel           acc=0.8726 macroF1=0.8638 fake P/R 0.891/0.775  (2686/3078)
  oracle           acc=0.8739 macroF1=0.8651 fake P/R 0.894/0.776  (2690/3078)
  two_bert         acc=0.8882 macroF1=0.8806 fake P/R 0.911/0.798  (2734/3078)
  oracle_two_bert  acc=0.8908 macroF1=0.8833 fake P/R 0.915/0.800  (2742/3078)
  monolithic       acc=0.8684 macroF1=0.8637 fake P/R 0.821/0.856  (2673/3078)

--- seed 42 (3078 test articles, always-majority baseline 0.6017) ---
  funnel           acc=0.8392 macroF1=0.8254 fake P/R 0.870/0.701  (2583/3078)
  oracle           acc=0.8596 macroF1=0.8484 fake P/R 0.892/0.737  (2646/3078)
  two_bert         acc=0.8671 macroF1=0.8573 fake P/R 0.892/0.759  (2669/3078)
  oracle_two_bert  acc=0.8804 macroF1=0.8717 fake P/R 0.909/0.777  (2710/3078)
  monolithic       acc=0.8697 macroF1=0.8640 fake P/R 0.838/0.834  (2677/3078)

--- seed 123 (3078 test articles, always-majority baseline 0.6017) -

## (Conditional) Featurize misrouted articles — GossipCop, binary-features-only case

Only needed if the evaluate cell above reported LIFE-missing fallbacks and you want the funnel's LIFE
branch to genuinely answer for misrouted human articles instead of falling back. This featurizes ONLY
the test articles that some router sent to LIFE without features (~a few hundred, minutes-to-an-hour):
Step 1 loads the existing GossipCop extractor checkpoint (`dataset/gossipcop/bert_bin.pt` — it skips
training when the file exists), then Steps 2–3 run the normal pipeline on the mini set.
**Afterwards: re-run the assemble → LIFE → evaluate cells above** (the assemble cell picks up the
extra features dir automatically).

In [ ]:
# Build the mini raw set: test articles routed to LIFE that lack features.
import json, os
from collections import defaultdict
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

needed = set()
for seed in (7, 42, 123):
    with open(f'{GC_RUN}/life_coverage_seed{seed}.json', encoding='utf-8') as f:
        missing = set(json.load(f)['missing_test_keys'])
    routed = set()
    with open(f'{GC_RUN}/preds_router_seed{seed}.jsonl', encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            if row['pred'] == 1:
                routed.add(row['key'])
    needed |= missing & routed
print(f'{len(needed)} unique test articles routed to LIFE without features (union over seeds)')

name_map = {'HF': ('HF_fake.jsonl', 'human_fake'), 'HR': ('HR_true.jsonl', 'human_true'),
            'MF': ('MF_fake.jsonl', 'gpt3.5_fake'), 'MR': ('MR_true.jsonl', 'gpt3.5_true')}
by_subset = defaultdict(list)
for key in needed:
    subset, idx = key.split(':')
    by_subset[subset].append(int(idx))

os.makedirs(GC_MISROUTE_RAW, exist_ok=True)
for subset, idxs in sorted(by_subset.items()):
    with open(f'{GOSSIPCOP_DIR}/{subset}.json', encoding='utf-8') as f:
        src = json.load(f)
    fname, label = name_map[subset]
    # sorted(idxs) keeps source order, so the new feature files stay an
    # ordered subsequence of their subset (required by the text join).
    with open(f'{GC_MISROUTE_RAW}/{fname}', 'w', encoding='utf-8') as f:
        for i in sorted(idxs):
            r = src[str(i)]
            f.write(json.dumps({'id': r['id'], 'text': r['text'], 'label': label},
                               ensure_ascii=False) + '\n')
    print(f'{subset}: {len(idxs)} articles -> {fname}')

In [ ]:
# Steps 1-3 on the mini set (extractor ckpt is LOADED, not retrained; top_k=15 = GossipCop's k)
!python dataset/1_keySentenceExtraction.py --data_dir "{GC_MISROUTE_RAW}" --output_file "{GC_MISROUTE_RAW}/key_sentences.jsonl" --top_k 15 --model_path "{PROJECT_DIR}/dataset/gossipcop/bert_bin.pt" --gpu 0
!python dataset/2_concate.py --folder_path "{GC_MISROUTE_RAW}" --important_sentences_file "{GC_MISROUTE_RAW}/key_sentences.jsonl"
!python dataset/3_gen_features_local.py --input_dir "{GC_MISROUTE_RAW}" --output_dir "{GC_MISROUTE_FEATURES}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16
# Now RE-RUN the GossipCop assemble -> LIFE -> evaluate cells above.

## Notes / caveats (read before quoting numbers)

- **Feature-space leak (inherited, flagged not fixed):** the Step-1 key-sentence extractor and the
  LLaMA features on Drive were generated ONCE over all articles, so LIFE's feature pipeline saw
  test-split articles. Only the classifiers (all five trained models) respect the split. A fully
  clean run would regenerate features per split — hours of A100 per seed; recorded as optional.
- **Group split shifts the anchors.** §7m/§7q used article-level splits; here a story and its
  GPT-3.5 rewrites travel together (id-groups), and the protocol is shared. Component anchors
  (router ~0.96 GC, human expert ~0.82 GC, LIFE ~0.87 PF) should land close but not identical —
  LARGE deviations mean a plumbing bug, not news.
- **LIFE protocol here vs §7e:** trains on the shared 70% (not its own 80/20 seed-0 split), and
  `--seed` matches the split seed. The per-epoch metrics train_bce.py prints are over the FULL mixed
  test set (including human articles LIFE never claims to handle) — ignore them; LIFE's own-slice
  anchor comes from evaluate_system.py.
- **Conventions:** veracity fake=1/real=0 everywhere in funnel_system (matches the LIFE BCE head;
  run_life_bert.py's HF-vs-HR track used the opposite — acc/macro-F1 comparisons are unaffected).
- **Label conflicts stay in the test set** (~0.8% of GossipCop bodies carry both labels, §7r): they
  are honest data noise, identical for every variant.
- `bce_en.pt` in the repo root is clobbered by every LIFE run — run cells sequentially, and don't
  run this notebook concurrently with the other LIFE notebooks.
- PolitiFact++ numbers are pilot/plumbing only (~85-article test, 1 article ≈ 1.2pp).